# WB — TFT + N-HiTS×5 + LGBM (полный пайплайн)

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'neuralforecast', 'lightgbm', 'xgboost'], check=True)
print('OK')

In [ ]:
import warnings, os, psutil, gc, glob
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import Ridge

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, TFT
from neuralforecast.losses.pytorch import MAE

try:
    import subprocess as _sp
    _gpu = _sp.check_output(['nvidia-smi','--query-gpu=index,name,memory.total','--format=csv,noheader'], text=True).strip()
    print('GPUs:', _gpu)
    GPU_OK = True
except:
    print('No GPU'); GPU_OK = False

print(f'torch={torch.__version__} | cuda={torch.cuda.is_available()}')

TARGET_COL  = 'target_1h'
FP          = 8
TRAIN_PATH  = '/kaggle/input/datasets/maratsaratov/wb-chall/train_solo_track.parquet'
TEST_PATH   = '/kaggle/input/datasets/maratsaratov/wb-chall/test_solo_track.parquet'
CKPT_HLD    = '/kaggle/working/ckpt_tft_holdout'
CKPT_FULL   = '/kaggle/working/ckpt_tft_full'
RANDOM_STATE = 42
FORECAST_POINTS = FP
FUTURE_COLS = [f'target_step_{i}' for i in range(1, FP+1)]
os.makedirs(CKPT_HLD,  exist_ok=True)
os.makedirs(CKPT_FULL, exist_ok=True)
print('OK')

## Monkey patch

In [ ]:
_orig_init = pl.Trainer.__init__

def _patched_init(self, *args, **kwargs):
    forbidden = {
        'lstm_layers','num_heads','attn_dropout','hidden_size','dropout',
        'futr_exog_list','learning_rate','max_steps','batch_size',
        'random_seed','enable_progress_bar','trainer_kwargs'
    }
    for k in forbidden:
        kwargs.pop(k, None)
    _orig_init(self, *args, **kwargs)

pl.Trainer.__init__ = _patched_init
print('Monkey patch применён.')

## 1. Данные

In [ ]:
def reduce_mem(df):
    for c in df.select_dtypes('float64').columns: df[c] = df[c].astype('float32')
    for c in df.select_dtypes('int64').columns:   df[c] = df[c].astype('int32')
    return df

train_df = reduce_mem(pd.read_parquet(TRAIN_PATH))
test_df  = reduce_mem(pd.read_parquet(TEST_PATH))
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])
test_df['timestamp']  = pd.to_datetime(test_df['timestamp'])
train_df = train_df.sort_values(['route_id','timestamp']).reset_index(drop=True)
test_df  = test_df.sort_values(['route_id','timestamp']).reset_index(drop=True)

STATUS_COLS  = sorted([c for c in train_df.columns if c.startswith('status_')])
INFERENCE_TS = train_df['timestamp'].max()

test_work = test_df.copy()
test_work['step_num'] = (
    (test_work['timestamp'] - INFERENCE_TS).dt.total_seconds() / 1800
).round().astype(int)

route_medians = train_df.groupby('route_id')[TARGET_COL].median().clip(lower=1.0)
print(f'INFERENCE_TS: {INFERENCE_TS}')
print(f'Train: {train_df.shape}')

## 2. Признаки для TFT/NHiTS

In [ ]:
FUTURE_COVS = ['hour_norm','minute_norm','dow_norm','is_saturday','time_slot_norm']

def add_time_feats(df):
    df = df.copy()
    df['hour_norm']      = (df['ds'].dt.hour / 24.0).astype('float32')
    df['minute_norm']    = (df['ds'].dt.minute / 60.0).astype('float32')
    df['dow_norm']       = (df['ds'].dt.dayofweek / 6.0).astype('float32')
    df['is_saturday']    = (df['ds'].dt.dayofweek == 5).astype('float32')
    df['time_slot_norm'] = ((df['ds'].dt.hour*2 + df['ds'].dt.minute//30) / 47.0).astype('float32')
    return df

df_base = train_df[['route_id','timestamp',TARGET_COL]].copy()
df_base.columns = ['unique_id','ds','y']
df_base['y'] = np.log1p(df_base['y'] / df_base['unique_id'].map(route_medians))

df_tft = add_time_feats(df_base.copy())

HOLDOUT_START = pd.Timestamp('2025-10-25 11:00:00')
HOLDOUT_END   = pd.Timestamp('2025-10-25 14:30:00')

df_base_fit = df_base[df_base['ds'] < HOLDOUT_START].copy()
df_tft_fit  = df_tft[df_tft['ds']   < HOLDOUT_START].copy()
df_hld      = df_base[(df_base['ds'] >= HOLDOUT_START) & (df_base['ds'] <= HOLDOUT_END)].copy()
df_hld['y_true']   = np.expm1(df_hld['y']) * df_hld['unique_id'].map(route_medians)
df_hld['step_num'] = df_hld.groupby('unique_id').cumcount() + 1

future_tft = add_time_feats(
    test_df[['route_id','timestamp']].rename(columns={'route_id':'unique_id','timestamp':'ds'})
)
holdout_ts  = pd.date_range(HOLDOUT_START, HOLDOUT_END, freq='30T')
future_hld  = add_time_feats(pd.DataFrame({
    'unique_id': np.repeat(df_base['unique_id'].unique(), len(holdout_ts)),
    'ds':        np.tile(holdout_ts, df_base['unique_id'].nunique())
}))

print(f'df_base_fit: {df_base_fit.shape} | Holdout: {df_hld.shape}')

## 3. Метрика

In [ ]:
class WapePlusRbias:
    def calculate(self, y_true, y_pred):
        yt = np.asarray(y_true,'float64').ravel()
        yp = np.asarray(y_pred,'float64').ravel()
        return float(np.abs(yp-yt).sum()/(yt.sum()+1e-9)+abs(yp.sum()/(yt.sum()+1e-9)-1))
    def components(self, y_true, y_pred):
        yt = np.asarray(y_true,'float64').ravel()
        yp = np.asarray(y_pred,'float64').ravel()
        return np.abs(yp-yt).sum()/(yt.sum()+1e-9), yp.sum()/(yt.sum()+1e-9)-1

metric = WapePlusRbias()

def eval_raw(raw_df, col, tag):
    df = raw_df.copy()
    df['pred'] = np.expm1(df[col]) * df['unique_id'].map(route_medians)
    df['pred'] = df['pred'].clip(0, None)
    ev = df[['unique_id','ds','pred']].merge(df_hld[['unique_id','ds','y_true']], on=['unique_id','ds'])
    ev['step'] = ev.groupby('unique_id').cumcount() + 1
    sc = metric.calculate(ev['y_true'].values, ev['pred'].values)
    w, rb = metric.components(ev['y_true'].values, ev['pred'].values)
    steps = [metric.calculate(ev.loc[ev['step']==s,'y_true'], ev.loc[ev['step']==s,'pred'])
             for s in range(1, FP+1)]
    print(f'[{tag}] {sc:.4f} | WAPE={w:.4f} | RBias={rb:+.4f}')
    print(f'  steps: {" ".join(f"{x:.3f}" for x in steps)}')
    return sc, ev

print('OK')

## 4. Feature Engineering для LGBM

In [ ]:
def compute_profiles(df, tc):
    route_p = (
        df.groupby('route_id')[tc]
        .agg(route_mean='mean', route_std='std', route_median='median',
             route_q10=lambda x: x.quantile(.10),
             route_q25=lambda x: x.quantile(.25),
             route_q75=lambda x: x.quantile(.75),
             route_q90=lambda x: x.quantile(.90))
        .reset_index()
    )
    for c in route_p.columns:
        if c != 'route_id': route_p[c] = route_p[c].astype('float32')

    tmp = df.copy()
    tmp['time_slot'] = (tmp['timestamp'].dt.hour * 2 + tmp['timestamp'].dt.minute // 30).astype('int8')
    tmp['dayofweek'] = tmp['timestamp'].dt.dayofweek.astype('int8')

    slot_p = (
        tmp.groupby(['route_id','time_slot'])[tc].mean()
        .reset_index().rename(columns={tc: 'route_slot_mean'})
    )
    dow_p = (
        tmp.groupby(['route_id','dayofweek'])[tc].mean()
        .reset_index().rename(columns={tc: 'route_dow_mean'})
    )
    sat_df = tmp[tmp['dayofweek'] == 5]
    sat_p = (
        sat_df.groupby(['route_id','time_slot'])[tc].mean()
        .reset_index().rename(columns={tc: 'route_sat_slot_mean'})
    )
    recent_sat = df[df['timestamp'].dt.dayofweek == 5].copy()
    recent_sat['time_slot'] = (recent_sat['timestamp'].dt.hour * 2 +
                                recent_sat['timestamp'].dt.minute // 30).astype('int8')
    recent_sat['sat_week'] = (recent_sat['timestamp'].dt.date.apply(
        lambda d: d)).rank(method='dense').astype(int)
    max_sat_week = recent_sat['sat_week'].max()
    recent_sat_4w = recent_sat[recent_sat['sat_week'] >= max_sat_week - 4]
    sat_recent_p = (
        recent_sat_4w.groupby(['route_id','time_slot'])[tc].mean()
        .reset_index().rename(columns={tc: 'route_sat_recent_mean'})
    )
    cross_ts = (
        df.groupby('timestamp')[tc]
        .agg(cross_mean='mean', cross_std='std', cross_median='median')
        .reset_index()
    )
    for c in ['cross_mean','cross_std','cross_median']:
        cross_ts[c] = cross_ts[c].astype('float32')
    for p in [route_p, slot_p, dow_p, sat_p, sat_recent_p]:
        for c in p.columns:
            if c not in ('route_id','time_slot','dayofweek'):
                p[c] = p[c].astype('float32')
    return route_p, slot_p, dow_p, sat_p, sat_recent_p, cross_ts


def build_features(df, tc, route_p, slot_p, dow_p, sat_p, sat_recent_p, cross_ts, status_cols):
    df = df.sort_values(['route_id','timestamp']).reset_index(drop=True)
    ts = df['timestamp']
    df['hour']        = ts.dt.hour.astype('int8')
    df['minute']      = ts.dt.minute.astype('int8')
    df['dayofweek']   = ts.dt.dayofweek.astype('int8')
    df['dayofmonth']  = ts.dt.day.astype('int8')
    df['month']       = ts.dt.month.astype('int8')
    df['is_weekend']  = (df['dayofweek'] >= 5).astype('int8')
    df['is_saturday'] = (df['dayofweek'] == 5).astype('int8')
    df['time_slot']   = (df['hour']*2 + df['minute']//30).astype('int8')
    df['hour_sin']    = np.sin(2*np.pi*df['hour']/24).astype('float32')
    df['hour_cos']    = np.cos(2*np.pi*df['hour']/24).astype('float32')
    df['dow_sin']     = np.sin(2*np.pi*df['dayofweek']/7).astype('float32')
    df['dow_cos']     = np.cos(2*np.pi*df['dayofweek']/7).astype('float32')
    df['slot_sin']    = np.sin(2*np.pi*df['time_slot']/48).astype('float32')
    df['slot_cos']    = np.cos(2*np.pi*df['time_slot']/48).astype('float32')
    g = df.groupby('route_id', sort=False)[tc]
    for lag in [1,2,3,4,6,8,12,16,24,48,96,336]:
        df[f'lag_{lag}'] = g.shift(lag).astype('float32')
    for w in [4,8,16,48]:
        r = g.shift(1).rolling(w, min_periods=1)
        df[f'rmean_{w}'] = r.mean().astype('float32')
        df[f'rstd_{w}']  = r.std().astype('float32')
        df[f'rmax_{w}']  = r.max().astype('float32')
        df[f'rmin_{w}']  = r.min().astype('float32')
    for span in [4,12,24]:
        df[f'ewm_{span}'] = (
            g.shift(1).transform(lambda x: x.ewm(span=span, min_periods=1).mean())
            .astype('float32')
        )
    df['diff_1']  = g.diff(1).astype('float32')
    df['diff_4']  = g.diff(4).astype('float32')
    df['diff_8']  = g.diff(8).astype('float32')
    df['diff_16'] = g.diff(16).astype('float32')
    df['diff_48'] = g.diff(48).astype('float32')
    def linear_trend(x):
        def _trend(s):
            vals = s.values
            n = min(8, len(vals))
            if n < 2: return 0.0
            y = vals[-n:]
            mask = ~np.isnan(y)
            if mask.sum() < 2: return 0.0
            x_t = np.arange(n)[mask].astype('float32')
            y_t = y[mask]
            x_m, y_m = x_t.mean(), y_t.mean()
            denom = ((x_t - x_m)**2).sum()
            if denom == 0: return 0.0
            return float(((x_t-x_m)*(y_t-y_m)).sum() / denom)
        return x.expanding(2).apply(_trend, raw=False)
    df['target_trend'] = g.shift(1).transform(linear_trend).astype('float32')
    for st in status_cols:
        gs = df.groupby('route_id', sort=False)[st]
        for lag in [1,4,8,16,24,48]:
            df[f'{st}_lag_{lag}'] = gs.shift(lag).astype('float32')
        for w in [4,8,16]:
            r = gs.shift(1).rolling(w, min_periods=1)
            df[f'{st}_rmean_{w}'] = r.mean().astype('float32')
            df[f'{st}_rstd_{w}']  = r.std().astype('float32')
        df[f'{st}_diff_1']  = gs.diff(1).astype('float32')
        df[f'{st}_diff_4']  = gs.diff(4).astype('float32')
    df = df.merge(route_p,      on='route_id',               how='left')
    df = df.merge(slot_p,       on=['route_id','time_slot'],  how='left')
    df = df.merge(dow_p,        on=['route_id','dayofweek'],  how='left')
    df = df.merge(sat_p,        on=['route_id','time_slot'],  how='left')
    df = df.merge(sat_recent_p, on=['route_id','time_slot'],  how='left')
    df = df.merge(cross_ts,     on='timestamp',               how='left')
    df['dev_from_mean']     = (df[tc] - df['route_mean']).astype('float32')
    df['rel_dev']           = (df[tc] / (df['route_mean']+1)).astype('float32')
    df['dev_from_slot']     = (df[tc] - df['route_slot_mean']).astype('float32')
    df['dev_from_cross']    = (df[tc] - df['cross_mean']).astype('float32')
    df['target_norm']       = (df[tc] / (df['route_mean']+1)).astype('float32')
    df['dev_from_sat']      = (df[tc] - df['route_sat_slot_mean'].fillna(df['route_slot_mean'])).astype('float32')
    df['dev_from_sat_rec']  = (df[tc] - df['route_sat_recent_mean'].fillna(df['route_slot_mean'])).astype('float32')
    for lag in [1,4,8]:
        col = f'lag_{lag}'
        if col in df.columns:
            df[f'{col}_norm'] = (df[col] / (df['route_mean']+1)).astype('float32')
    df['hour_mean_inter'] = (df['hour_sin'] * df['route_mean']).astype('float32')
    df['dow_std_inter']   = (df['dow_cos']  * df['route_std']).astype('float32')
    return df

print('FE функции готовы.')

In [ ]:
%%time
route_p, slot_p, dow_p, sat_p, sat_recent_p, cross_ts = compute_profiles(train_df, TARGET_COL)
train_feat = build_features(train_df, TARGET_COL,
                            route_p, slot_p, dow_p, sat_p, sat_recent_p, cross_ts,
                            STATUS_COLS)
print(f'train_feat: {train_feat.shape}')

In [ ]:
rg = train_feat.groupby('route_id', sort=False)[TARGET_COL]
for step in range(1, FORECAST_POINTS+1):
    train_feat[f'target_step_{step}'] = rg.shift(-step).astype('float32')

inference_feat = (
    train_feat[train_feat['timestamp'] == INFERENCE_TS]
    .drop_duplicates('route_id').copy()
)
supervised_df = train_feat.dropna(subset=FUTURE_COLS).copy()
del train_feat; gc.collect()
print(f'supervised_df: {supervised_df.shape}')

## 5. Val split и подготовка X/y

In [ ]:
exclude       = {TARGET_COL,'timestamp','id',*FUTURE_COLS}
feature_cols  = [c for c in supervised_df.columns if c not in exclude]
cat_cols      = [c for c in feature_cols if c.endswith('_id')]

ts_max    = supervised_df['timestamp'].max()
val_start = ts_max - pd.Timedelta(days=7)
fit_df    = supervised_df[supervised_df['timestamp'] <  val_start].copy()
valid_df  = supervised_df[supervised_df['timestamp'] >= val_start].copy()
del supervised_df; gc.collect()

feat_no_id  = [c for c in feature_cols if c != 'route_id']
X_test_raw  = test_work[['id','route_id','timestamp','step_num']].merge(
    inference_feat[['route_id']+feat_no_id], on='route_id', how='left')
X_test_base = X_test_raw[feature_cols].copy()

masks_fit, masks_val, y_fit_s, y_val_s = {}, {}, {}, {}
for step in range(1, FORECAST_POINTS+1):
    col = f'target_step_{step}'
    mf  = fit_df[col].notna()
    mv  = valid_df[col].notna()
    masks_fit[step] = mf;  y_fit_s[step] = fit_df.loc[mf, col].values.astype('float32')
    masks_val[step] = mv;  y_val_s[step] = valid_df.loc[mv, col].values.astype('float32')

all_y_val = np.concatenate([y_val_s[s] for s in range(1, FORECAST_POINTS+1)])
print(f'fit: {fit_df.shape} | valid: {valid_df.shape}')
print(f'X_test_base: {X_test_base.shape}')

## 6. Параметры LGBM

In [ ]:
LGBM_P = {
    'objective':          'regression_l1',
    'metric':             'mae',
    'learning_rate':      0.04,
    'num_leaves':         255,
    'min_child_samples':  30,
    'feature_fraction':   0.8,
    'bagging_fraction':   0.8,
    'bagging_freq':       5,
    'reg_alpha':          0.1,
    'reg_lambda':         0.2,
    'device':             'gpu' if GPU_OK else 'cpu',
    'gpu_platform_id':    0,
    'gpu_device_id':      0,
    'max_bin':            63,
    'min_data_in_bin':    3,
    'seed':               RANDOM_STATE,
    'num_threads':        2,
    'verbosity':          -1,
}

def train_lgbm_step(step, params=LGBM_P, n_rounds=5000, patience=200, gpu_id=0):
    p = {**params, 'gpu_device_id': gpu_id}
    X_fit_step = fit_df.loc[masks_fit[step], feature_cols]
    X_val_step = valid_df.loc[masks_val[step], feature_cols]
    dtrain = lgb.Dataset(X_fit_step, label=y_fit_s[step], free_raw_data=True)
    dvalid = lgb.Dataset(X_val_step, label=y_val_s[step], reference=dtrain, free_raw_data=True)
    del X_fit_step; gc.collect()
    booster = lgb.train(
        p, dtrain, num_boost_round=n_rounds, valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(patience, verbose=False), lgb.log_evaluation(500)]
    )
    vp = booster.predict(X_val_step)
    tp = booster.predict(X_test_base)
    sc = metric.calculate(y_val_s[step], vp)
    print(f'  LGBM step {step}: iter={booster.best_iteration:4d} | val={sc:.4f}')
    del booster, dtrain, dvalid, X_val_step; gc.collect()
    return vp, tp, sc

print('OK')

## 7. Обучение LGBM

In [ ]:
%%time
val_lgbm, tst_lgbm, sc_lgbm = {},{},{}
for step in range(1, FORECAST_POINTS+1):
    val_lgbm[step], tst_lgbm[step], sc_lgbm[step] = train_lgbm_step(step, gpu_id=0)

lgbm_scores = [sc_lgbm[s] for s in range(1, FORECAST_POINTS+1)]
lgbm_total  = metric.calculate(all_y_val,
    np.concatenate([val_lgbm[s] for s in range(1, FORECAST_POINTS+1)]))
print(f'\nLGBM total: {lgbm_total:.4f}')
print(f'steps: {" ".join(f"{s:.3f}" for s in lgbm_scores)}')

## 8. Субботний бейзлайн и финальные предсказания LGBM

In [ ]:
sat_baseline = X_test_base['route_sat_recent_mean'].fillna(
    X_test_base['route_sat_slot_mean']).fillna(
    X_test_base['route_slot_mean']).values

val_baseline_preds = {}
for step in range(1, FORECAST_POINTS+1):
    X_v = valid_df.loc[masks_val[step], feature_cols]
    bl  = X_v['route_sat_recent_mean'].fillna(
          X_v['route_sat_slot_mean']).fillna(
          X_v['route_slot_mean']).values
    val_baseline_preds[step] = bl

tst_bst = {s: tst_lgbm[s] for s in range(1, FP+1)}

val_lgbm_hld = {}
hld_ts_sorted = train_df.sort_values(['route_id','timestamp']).reset_index(drop=True)
hld_mask_all  = (hld_ts_sorted['timestamp'] >= HOLDOUT_START) & \
                (hld_ts_sorted['timestamp'] <= HOLDOUT_END)
hld_idx = np.where(hld_mask_all.values)[0]
for step in range(1, FP+1):
    vl = val_lgbm[step]
    n  = min(len(hld_idx), len(vl))
    val_lgbm_hld[step] = vl[hld_idx[:n]] if n > 0 else np.array([])

bst_hld = val_lgbm_hld
print('LGBM holdout OK')
print(f'sat_baseline mean: {sat_baseline.mean():.0f}')

## 9. TFT holdout — с чекпоинтами

In [ ]:
%%time

def make_tft(seed=42, max_steps=500, ckpt_dir=None):
    callbacks = [
        EarlyStopping(monitor='valid_loss', patience=10, mode='min', verbose=True),
    ]
    if ckpt_dir:
        callbacks.append(ModelCheckpoint(
            dirpath=ckpt_dir,
            filename='step{step:05d}-loss{valid_loss:.4f}',
            every_n_train_steps=50,
            save_top_k=-1,
            save_last=True,
            monitor='valid_loss',
            mode='min',
        ))
    return TFT(
        h=FP, input_size=96, hidden_size=32, lstm_layers=2, num_heads=4,
        attn_dropout=0.1, dropout=0.1, futr_exog_list=FUTURE_COVS,
        loss=MAE(), learning_rate=1e-3, max_steps=max_steps, batch_size=8,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1, random_seed=seed, enable_progress_bar=True,
        trainer_kwargs={'precision': '16-mixed', 'callbacks': callbacks}
    )

def find_ckpt(ckpt_dir):
    ckpts = glob.glob(os.path.join(ckpt_dir, '*.ckpt'))
    if not ckpts:
        return None
    return max(ckpts, key=os.path.getmtime)

ckpt = find_ckpt(CKPT_HLD)
if ckpt:
    print(f'Чекпоинт найден: {os.path.basename(ckpt)}')
    nf_tft_hld = NeuralForecast.load(path=CKPT_HLD)
    print('Загружено!')
else:
    print('Обучаем TFT holdout...')
    nf_tft_hld = NeuralForecast(
        models=[make_tft(seed=42, max_steps=500, ckpt_dir=CKPT_HLD)], freq='30T')
    nf_tft_hld.fit(df=df_tft_fit, val_size=FP)
    nf_tft_hld.save(path=CKPT_HLD, overwrite=True)

In [ ]:
raw_tft_hld = nf_tft_hld.predict(futr_df=future_hld)
sc_tft, ev_tft = eval_raw(raw_tft_hld, 'TFT', 'TFT-holdout')

## 10. N-HiTS × 5 seeds

In [ ]:
def make_nhits(seed):
    return NHITS(
        h=FP, input_size=672,
        n_blocks=[1,1,1],
        mlp_units=[[256,256],[256,256],[256,256]],
        n_pool_kernel_size=[16,4,1],
        n_freq_downsample=[48,8,1],
        loss=MAE(), learning_rate=5e-4, max_steps=1500,
        batch_size=16, dropout_prob_theta=0.15,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1, random_seed=seed, enable_progress_bar=True,
    )

nhits_hld_preds, nhits_hld_scores = [], []

for seed in [42, 43, 44, 45, 46]:
    print(f'N-HiTS seed={seed}...')
    nf = NeuralForecast(models=[make_nhits(seed)], freq='30T')
    nf.fit(df=df_base_fit, val_size=FP)
    raw = nf.predict(df=df_base_fit)
    sc, ev = eval_raw(raw, 'NHITS', f'NHITS-s{seed}')
    nhits_hld_preds.append(ev[['unique_id','ds','pred']].rename(columns={'pred':f'pred_s{seed}'}))
    nhits_hld_scores.append(sc)
    del nf; gc.collect()

nhits_ens_hld = nhits_hld_preds[0].copy()
for p in nhits_hld_preds[1:]:
    nhits_ens_hld = nhits_ens_hld.merge(p, on=['unique_id','ds'])
pred_cols = [c for c in nhits_ens_hld.columns if c.startswith('pred_')]
nhits_ens_hld['pred'] = nhits_ens_hld[pred_cols].mean(axis=1)

ev_ens = nhits_ens_hld.merge(df_hld[['unique_id','ds','y_true']], on=['unique_id','ds'])
ev_ens['step'] = ev_ens.groupby('unique_id').cumcount() + 1
sc_ens = metric.calculate(ev_ens['y_true'].values, ev_ens['pred'].values)
w_ens, rb_ens = metric.components(ev_ens['y_true'].values, ev_ens['pred'].values)
steps_ens = [metric.calculate(ev_ens.loc[ev_ens['step']==s,'y_true'],
                               ev_ens.loc[ev_ens['step']==s,'pred'])
             for s in range(1, FP+1)]
print(f'N-HiTS seeds: {[f"{s:.4f}" for s in nhits_hld_scores]}')
print(f'N-HiTS ансамбль: {sc_ens:.4f} | WAPE={w_ens:.4f} | RBias={rb_ens:+.4f}')
print(f'  steps: {" ".join(f"{x:.3f}" for x in steps_ens)}')

## 11. Подбор каскада на holdout

In [ ]:
def cascade_holdout(step_source):
    all_preds, all_true = [], []
    for step in range(1, FP+1):
        mask = ev_ens['step'] == step
        yt   = ev_ens.loc[mask, 'y_true'].values
        src  = step_source[step]
        if src == 'lgbm':
            n  = min(len(bst_hld[step]), len(yt))
            yp = bst_hld[step][:n] if n == len(yt) else np.full(len(yt), bst_hld[step].mean())
        elif src == 'tft':
            yp = ev_tft.loc[ev_tft['step']==step,'pred'].values[:len(yt)]
        else:
            yp = ev_ens.loc[mask,'pred'].values
        all_preds.append(yp); all_true.append(yt)
    return metric.calculate(np.concatenate(all_true), np.concatenate(all_preds))

strategies = {
    'NHiTS only':                    {s:'nhits' for s in range(1,9)},
    'TFT only':                      {s:'tft'   for s in range(1,9)},
    'LGBM[1]+NHiTS[2-8]':           {1:'lgbm', **{s:'nhits' for s in range(2,9)}},
    'LGBM[1-2]+NHiTS[3-8]':         {**{s:'lgbm' for s in range(1,3)}, **{s:'nhits' for s in range(3,9)}},
    'LGBM[1]+TFT[2-5]+NHiTS[6-8]':  {1:'lgbm',2:'tft',3:'tft',4:'tft',5:'tft',6:'nhits',7:'nhits',8:'nhits'},
    'LGBM[1-2]+TFT[3-5]+NHiTS[6-8]':{1:'lgbm',2:'lgbm',3:'tft',4:'tft',5:'tft',6:'nhits',7:'nhits',8:'nhits'},
    'TFT[1-5]+NHiTS[6-8]':          {**{s:'tft' for s in range(1,6)},**{s:'nhits' for s in range(6,9)}},
    'LGBM[1]+TFT[2-8]':             {1:'lgbm',**{s:'tft' for s in range(2,9)}},
}

per_step = {}
for step in range(1, FP+1):
    b_src, b_sc = 'nhits', 999
    for src in ['lgbm','tft','nhits']:
        sc2 = cascade_holdout({**{s:'nhits' for s in range(1,9)}, step:src})
        if sc2 < b_sc: b_sc, b_src = sc2, src
    per_step[step] = b_src
strategies['Best per step'] = per_step

print(f'NHiTS ens: {sc_ens:.4f} | TFT: {sc_tft:.4f}\n')
results = {}
for name, strat in strategies.items():
    sc2 = cascade_holdout(strat)
    results[name] = sc2
    print(f'  {name:45s}: {sc2:.4f}')

best_name  = min(results, key=results.get)
best_strat = strategies[best_name]
print(f'\n Best: {best_name} = {results[best_name]:.4f}')
print(f'  По шагам: {best_strat}')

## 12. TFT финальная — с чекпоинтами

In [ ]:
%%time

ckpt_f = find_ckpt(CKPT_FULL)
if ckpt_f:
    print(f'Чекпоинт финальной TFT: {os.path.basename(ckpt_f)}')
    nf_tft_final = NeuralForecast.load(path=CKPT_FULL)
    print('Загружено!')
else:
    print('Обучаем финальную TFT...')
    nf_tft_final = NeuralForecast(
        models=[make_tft(seed=42, max_steps=500, ckpt_dir=CKPT_FULL)], freq='30T')
    nf_tft_final.fit(df=df_tft, val_size=FP)
    nf_tft_final.save(path=CKPT_FULL, overwrite=True)
    print(f'Сохранено: {CKPT_FULL}')

raw_tft_tst = nf_tft_final.predict(futr_df=future_tft)
raw_tft_tst['pred'] = np.expm1(raw_tft_tst['TFT']) * raw_tft_tst['unique_id'].map(route_medians)
raw_tft_tst['pred'] = raw_tft_tst['pred'].clip(0, None)
raw_tft_tst['step_num'] = raw_tft_tst.groupby('unique_id').cumcount() + 1
raw_tft_tst = raw_tft_tst.rename(columns={'unique_id':'route_id'})
tft_map = raw_tft_tst.set_index(['route_id','step_num'])['pred'].to_dict()
print(f'TFT test: mean={raw_tft_tst["pred"].mean():.0f}')

## 13. N-HiTS × 5 seeds финальное

In [ ]:
%%time
nhits_tst_preds = []

for seed in [42, 43, 44, 45, 46]:
    print(f'seed={seed}...')
    nf = NeuralForecast(models=[make_nhits(seed)], freq='30T')
    nf.fit(df=df_base)
    raw = nf.predict(df=df_base)
    raw['pred'] = np.expm1(raw['NHITS']) * raw['unique_id'].map(route_medians)
    raw['pred'] = raw['pred'].clip(0, None)
    raw['step_num'] = raw.groupby('unique_id').cumcount() + 1
    raw = raw.rename(columns={'unique_id':'route_id'})
    nhits_tst_preds.append(raw[['route_id','step_num','pred']].rename(columns={'pred':f'pred_s{seed}'}))
    del nf; gc.collect()

nhits_tst_ens = nhits_tst_preds[0].copy()
for p in nhits_tst_preds[1:]:
    nhits_tst_ens = nhits_tst_ens.merge(p, on=['route_id','step_num'])
pred_cols = [c for c in nhits_tst_ens.columns if c.startswith('pred_')]
nhits_tst_ens['pred_ens'] = nhits_tst_ens[pred_cols].mean(axis=1)
nhits_map = nhits_tst_ens.set_index(['route_id','step_num'])['pred_ens'].to_dict()
print(f'NHiTS ens test: mean={nhits_tst_ens["pred_ens"].mean():.0f}')

## 14. Сабмиты

In [ ]:
NHITS_CAL = 1.010
best_c_tft, best_sc_tft = 1.0, sc_tft
for c in np.arange(0.90, 1.11, 0.005):
    sc_c = metric.calculate(ev_tft['y_true'].values, ev_tft['pred'].values * c)
    if sc_c < best_sc_tft: best_c_tft, best_sc_tft = c, sc_c
print(f'TFT cal: {sc_tft:.4f} → {best_sc_tft:.4f} (c={best_c_tft:.3f})')

def make_sub(arr, name):
    sub = pd.DataFrame({'id':test_work['id'].values,
                        'y_pred':np.clip(arr,0,None)}).sort_values('id').reset_index(drop=True)
    assert len(sub)==8000
    sub.to_csv(f'/kaggle/working/submission_solo_{name}.csv', index=False)
    print(f'{name}: mean={sub["y_pred"].mean():.0f}')
    return sub

def build(strat, nhits_sc=1.0, tft_sc=1.0):
    final = np.zeros(len(test_work))
    for i in range(len(test_work)):
        step  = int(test_work.iloc[i]['step_num'])
        route = int(test_work.iloc[i]['route_id'])
        src = strat[step]
        if src == 'lgbm':   final[i] = tst_bst[step][i]
        elif src == 'tft':  final[i] = tft_map.get((route,step),0) * tft_sc
        else:               final[i] = nhits_map.get((route,step),0) * nhits_sc
    return final

def build_w(wl, wt, wn, nhits_sc=1.0):
    final = np.zeros(len(test_work))
    for i in range(len(test_work)):
        step  = int(test_work.iloc[i]['step_num'])
        route = int(test_work.iloc[i]['route_id'])
        final[i] = (wl[step]*tst_bst[step][i] +
                    wt[step]*tft_map.get((route,step),0)*best_c_tft +
                    wn[step]*nhits_map.get((route,step),0)*nhits_sc)
    return final

sub1 = make_sub(build({s:'nhits' for s in range(1,9)}, NHITS_CAL), 'nhits_ens5')
sub2 = make_sub(build({s:'tft' for s in range(1,9)}, tft_sc=best_c_tft), 'tft_cal')
sub3 = make_sub(build(best_strat, NHITS_CAL, best_c_tft), 'cascade_best')
sub4 = make_sub(build({1:'lgbm',2:'tft',3:'tft',4:'tft',5:'tft',6:'nhits',7:'nhits',8:'nhits'},
                      NHITS_CAL, best_c_tft), 'lgbm1_tft25_nhits68')
sub5 = make_sub(build({1:'lgbm',2:'lgbm',**{s:'nhits' for s in range(3,9)}}, NHITS_CAL),
                'lgbm12_nhits38')

WL = {1:1.0,2:0.5,3:0.0,4:0.0,5:0.0,6:0.0,7:0.0,8:0.0}
WT = {1:0.0,2:0.3,3:0.6,4:0.6,5:0.4,6:0.2,7:0.0,8:0.0}
WN = {s:1.0-WL[s]-WT[s] for s in range(1,9)}
sub6 = make_sub(build_w(WL, WT, WN, NHITS_CAL), 'cascade_soft')

print(f'\n{"-"*55}')
print(f'N-HiTS x5:    {sc_ens:.4f}')
print(f'TFT:          {sc_tft:.4f} → cal {best_sc_tft:.4f} (c={best_c_tft:.3f})')
print(f'Best cascade: {results[best_name]:.4f}  ({best_name})')
print(f'RAM: {psutil.Process(os.getpid()).memory_info().rss/1e9:.2f} GB')

In [ ]:
for f in sorted(os.listdir('/kaggle/working')):
    if f.endswith('.csv'):
        sz = os.path.getsize(f'/kaggle/working/{f}')//1024
        print(f'  {f}  {sz} KB')

## Описание итоговой модели

### Задача

Прогнозирование загруженности маршрутов WB Solo Track на горизонте 8 шагов (4 часа, интервал 30 минут). Метрика: WAPE + |RBias| — сумма взвешенной абсолютной ошибки и отклонения суммарного прогноза от факта.

---

### Компоненты ансамбля

Итоговое решение представляет собой каскадный ансамбль трёх семейств моделей, каждая из которых специализируется на своей части горизонта прогноза.

**1. LightGBM (градиентный бустинг)**

Обучается отдельно для каждого из 8 шагов горизонта в режиме прямого многошагового прогноза (DIRECT strategy). Входные признаки: временны́е (час, день недели, тайм-слот в синус/косинус кодировке), лаги таргета (1, 2, 3, 4, 6, 8, 12, 16, 24, 48, 96, 336 шагов), скользящие средние и стандартные отклонения (окна 4/8/16/48), экспоненциальные скользящие средние (EWM), линейный тренд за последние 8 точек, статусные признаки с лагами и трендами, профили маршрута (среднее, медиана, квантили), профили по тайм-слоту и дню недели, субботний профиль (исторический и за последние 4 недели), кросс-маршрутные агрегаты по временно́й метке. Всего свыше 200 признаков. Параметры: MAE-оптимизация, learning_rate=0.04, num_leaves=255, GPU-ускорение, early stopping после 200 раундов без улучшения (до 5000 итераций). Валидация: последние 7 дней обучающей выборки. LGBM наиболее силён на коротком горизонте (step 1–2), где информация о текущем состоянии маршрута максимально ценна.

**2. Temporal Fusion Transformer (TFT)**

Нейронная сеть из библиотеки NeuralForecast, обрабатывающая временны́е ряды как последовательность. Архитектура: input_size=96 (48 часов ретроспективы), hidden_size=32, 2 LSTM-слоя, 4 головы attention, dropout=0.1, смешанная точность float16. Будущие ковариаты: нормализованные час, минута, день недели, флаг субботы, тайм-слот. Целевая переменная подаётся в log1p-нормализованном виде относительно медианы маршрута. Обучается с чекпоинтированием каждые 50 шагов — при повторном запуске ноутбука автоматически продолжается с последнего чекпоинта. TFT эффективен на среднем горизонте (step 2–5), где важны паттерны из ретроспективного окна и будущих ковариат.

**3. N-HiTS × 5 seeds**

Ансамбль из 5 нейронных сетей N-HiTS (Neural Hierarchical Interpolation for Time Series) с разными seed (42–46). Архитектура: input_size=672 (2 недели ретроспективы), 3 блока с пулингом [16, 4, 1] и даунсэмплингом частот [48, 8, 1], MLP [256, 256] в каждом блоке, dropout=0.15. Обучается без внешних ковариат только по значениям ряда. Предсказания 5 моделей усредняются. Применяется калибровочный множитель NHITS_CAL=1.010. N-HiTS устойчив на длинном горизонте (step 6–8).

---

### Каскадная стратегия

Оптимальное распределение источников по шагам подбирается на holdout-множестве (25 октября 2025, 11:00–14:30). Для каждого шага оценивается LGBM, TFT и NHiTS — выбирается лучший источник. Помимо автоматически найденного «best cascade», генерируются сабмиты с мягким взвешенным каскадом `cascade_soft`: step 1 — 100% LGBM; step 2 — 50% LGBM + 30% TFT + 20% NHiTS; step 3 — 60% TFT + 40% NHiTS; step 4–5 — смесь TFT и NHiTS; step 6–8 — преимущественно NHiTS. Именно сабмит `submission_solo_cascade_soft.csv` показал лучший результат на публичном лидерборде.

---

### Обработка субботних данных

Суббота — отдельный режим работы маршрутов. В качестве дополнительного сигнала используется субботний бейзлайн: среднее значение таргета по маршруту и тайм-слоту за субботы (с упором на последние 4 недели). Этот признак включён в LGBM как `route_sat_recent_mean` и `route_sat_slot_mean`, а также применяется как отдельный прогноз при анализе диагностики субботних строк.

---

### Пайплайн выполнения

1. Загрузка данных и feature engineering для LGBM (признаки, профили, лаги)
2. Обучение LGBM отдельно для каждого из 8 шагов
3. Формирование TFT/NHiTS входных данных (log1p-нормализация, временны́е ковариаты)
4. Обучение TFT holdout с чекпоинтами → оценка на holdout
5. Обучение N-HiTS × 5 seeds → оценка ансамбля на holdout
6. Подбор оптимального каскада по holdout-метрике
7. Обучение TFT на всём train (с чекпоинтами)
8. Обучение N-HiTS × 5 seeds на всём train
9. Генерация 6 вариантов сабмита: nhits_ens5, tft_cal, cascade_best, lgbm1_tft25_nhits68, lgbm12_nhits38, cascade_soft